In [ ]:
!kaggle datasets download -d dubradave/hospital-readmissions -p ./data --unzip

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("./data/hospital_readmissions.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df['readmitted'] = df['readmitted'].map({'no': 0, 'yes': 1})

In [ ]:
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
X = df.drop('readmitted', axis=1)
y = df['readmitted']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
auc = roc_auc_score(y_test, y_proba)
print("ROC-AUC:", auc)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - 30-Day Readmission Prediction")
plt.legend()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
tn, fp, fn, tp = cm.ravel()
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cost_fn_per_case = 15000
cost_fp_per_case = 1500

total_fn_cost = fn * cost_fn_per_case
total_fp_cost = fp * cost_fp_per_case

print("Estimated cost of False Negatives (missed readmissions):", total_fn_cost)
print("Estimated cost of False Positives (unnecessary interventions):", total_fp_cost)
print("Total estimated cost:", total_fn_cost + total_fp_cost)

In [ ]:
coeffs = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

coeffs